In [17]:
# Step 1: Import Libraries and Load the Model
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [18]:
# Load the IMDB dataset word index
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

In [19]:
# Load the pre-trained model with ReLU activation
model = load_model('simple_rnn_imdb.h5')
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 100, 128)          1280000   
                                                                 
 simple_rnn_1 (SimpleRNN)    (None, 32)                5152      
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 1285185 (4.90 MB)
Trainable params: 1285185 (4.90 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [20]:
model.get_weights()

[array([[ 0.07756172, -0.00727805,  0.03655145, ...,  0.02366537,
         -0.01701648, -0.01716802],
        [ 0.06897149, -0.04010013,  0.03095522, ..., -0.0591415 ,
          0.02577513,  0.04801748],
        [ 0.04517128,  0.03986862,  0.02942194, ...,  0.03114883,
         -0.00650293, -0.01225228],
        ...,
        [ 0.019545  ,  0.01631349,  0.00873407, ...,  0.0579842 ,
         -0.04249813, -0.04795015],
        [ 0.03981122,  0.03464273,  0.01420821, ...,  0.01823471,
         -0.02848379,  0.01846782],
        [-0.01808   , -0.05178285, -0.05357855, ...,  0.06311367,
          0.01649955,  0.03222372]], dtype=float32),
 array([[-0.07397792, -0.18169588, -0.23054732, ...,  0.08334569,
          0.08598622, -0.00826047],
        [ 0.20478423,  0.21153066, -0.17673756, ..., -0.04040768,
          0.1617995 , -0.13533136],
        [ 0.23887044,  0.07213492,  0.06481081, ...,  0.10688967,
          0.18256342, -0.0756292 ],
        ...,
        [-0.0617357 ,  0.14463407,  0.0

In [21]:
# Step 2: Helper Functions
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review]) ##decode the sample review by mapping each integer back to its corresponding word using the reverse_word_index dictionary. The get() method is used to retrieve the word for each integer index, and if an index is not found in the dictionary (e.g., due to reserved indices for padding or special tokens), it returns a '?' character. The resulting list of words is then joined into a single string to form the decoded review.

# Function to preprocess user input
def preprocess_text(text):
    words = text.lower().split() ##convert the input text to lowercase and split it into individual words. This is a common preprocessing step in natural language processing (NLP) tasks, as it helps to standardize the text and make it easier to work with. By converting the text to lowercase, we ensure that words like "Good" and "good" are treated as the same word, and by splitting the text into words, we can process each word individually for encoding.
    encoded_review = [word_index.get(word, 2) + 3 for word in words] ##encode the input review by mapping each word to its corresponding integer index using the word_index dictionary. The get() method is used to retrieve the index for each word, and if a word is not found in the dictionary (e.g., because it is not in the training vocabulary), it returns 2, which is a reserved index for unknown words. We add 3 to the retrieved index to account for reserved indices (0 for padding, 1 for start of sequence, and 2 for unknown words) in the IMDB dataset. The resulting list of integer indices represents the encoded review that can be fed into the model for prediction.
    padded_review = sequence.pad_sequences([encoded_review], maxlen=100) ##pad the encoded review to ensure that it has a consistent length of 200 words. The pad_sequences function takes the list of encoded reviews (in this case, a single review wrapped in a list) and pads it with zeros at the beginning if it is shorter than 200 words, or truncates it if it is longer than 200 words. This allows us to create a consistent input shape for our neural network model, which is necessary for making predictions.
    return padded_review

In [22]:
### Prediction  function

def predict_sentiment(review):
    preprocessed_input=preprocess_text(review) #preprocess the input review using the preprocess_text function, which encodes the review into a sequence of integer indices and pads it to ensure a consistent length of 200 words. The resulting preprocessed_input is a 2D array that can be fed into the model for prediction.

    prediction=model.predict(preprocessed_input) #make a prediction using the model's predict() method, which takes the preprocessed input and returns a prediction. The output is a 2D array where the first dimension corresponds to the batch size (in this case, 1 since we are predicting for a single review) and the second dimension corresponds to the predicted probability of the review being positive (a value between 0 and 1).

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    
    return sentiment, prediction[0][0]

In [23]:
# Step 4: User Input and Prediction
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment,score=predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

2026-04-23 12:52:07.303353: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 [==============================] - 1s 1s/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction Score: 0.5508897304534912
